In [5]:
import os
from dotenv import load_dotenv

load_dotenv()
QDRANT_API_KEY = os.getenv('qdrant_api_key')
QDRANT_URL = os.getenv('qdrant_url')
EURI_API_KEY = os.getenv('euron_api_key')

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=EURI_API_KEY,
    base_url="https://api.euron.one/api/v1/euri",
)

resp = client.embeddings.create(
    model="gemini-embedding-2-preview",
    input=["The quick brown fox jumps over the lazy dog."],
)
print(f"Dimensions: {len(resp.data[0].embedding)}")
print(f"First 5 values: {resp.data[0].embedding[:5]}")

Dimensions: 3072
First 5 values: [0.003027868689969182, -0.00013053484144620597, -0.003910216502845287, -0.01594909466803074, 0.0009256076882593334]


In [8]:
import os
from qdrant_client import QdrantClient
client = QdrantClient(url=QDRANT_URL,api_key = QDRANT_API_KEY)

In [15]:
from qdrant_client.models import Distance, VectorParams, PointStruct
COLLECTION_NAME = "euron_knowledge_base"

In [14]:
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config = VectorParams(
        size=len(resp.data[0].embedding), 
        distance=Distance.COSINE
    )
)


True

In [17]:
point = PointStruct(
    id=1, 
    vector=resp.data[0].embedding, 
    payload={
        "text": "The quick brown fox jumps over the lazy dog."
    }
)

In [18]:
client.upsert(
    collection_name=COLLECTION_NAME,
    points=[point]
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [19]:
documents = [
    {
        "text":
        """
        Machine learning enables computers to learn
        patterns from historical datasets.
        """,
        "title": "Machine Learning Basics",
        "course": "Machine Learning",
        "chapter": 1
    },
    {
        "text":
        """
        Deep learning uses artificial neural networks
        containing multiple layers.
        """,
        "title": "Deep Learning Introduction",
        "course": "Deep Learning",
        "chapter": 1
    },
    {
        "text":
        """
        MongoDB is a document-oriented NoSQL database
        that stores data using BSON documents.
        """,
        "title": "MongoDB Basics",
        "course": "Database Engineering",
        "chapter": 3
    },
    {
        "text":
        """
        Docker containers package applications together
        with their dependencies.
        """,
        "title": "Docker Fundamentals",
        "course": "DevOps",
        "chapter": 2
    },
    {
        "text":
        """
        Retrieval augmented generation retrieves relevant
        information before generating an LLM response.
        """,
        "title": "RAG Fundamentals",
        "course": "Generative AI",
        "chapter": 5
    }
]

In [21]:
def generate_embedding(text):
    client = OpenAI(
        api_key=EURI_API_KEY,
        base_url="https://api.euron.one/api/v1/euri",
    )

    resp = client.embeddings.create(
        model="gemini-embedding-2-preview",
        input=text,
    )
    return resp.data[0].embedding

In [23]:
for i in documents:
    point = PointStruct(
        id=i["chapter"],
        vector=generate_embedding(i["text"]),
        payload={
            "text": i["text"],
            "title": i["title"],
            "chapter": i["chapter"],
            "course": i["course"]
        }
    )

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=[point]
    )

In [33]:
query = "Tell me about mongo db"
query_embedding = generate_embedding(query)

In [34]:
query_embedding

[0.01253670733422041,
 0.0283940639346838,
 0.00758250430226326,
 0.010694852098822594,
 0.009139510802924633,
 -0.02197888307273388,
 -0.021241581067442894,
 0.014605371281504631,
 0.009094786830246449,
 -0.05169558525085449,
 0.007667182479053736,
 0.014258780516684055,
 -0.01510068029165268,
 -0.013291501440107822,
 0.027936195954680443,
 -0.009440230205655098,
 0.03755202889442444,
 -0.006295376922935247,
 -0.004988383501768112,
 -0.002824971219524741,
 0.01609482802450657,
 -0.01463291235268116,
 -0.0019926277454942465,
 0.02829969860613346,
 -0.0005286871455609798,
 0.03516589105129242,
 -0.024596450850367546,
 -0.011849832721054554,
 -0.007120972033590078,
 0.12566791474819183,
 -0.03541186451911926,
 -0.012929483316838741,
 0.0030327800195664167,
 -0.007218117360025644,
 0.000580238236580044,
 0.017667951062321663,
 0.006060346961021423,
 0.026561936363577843,
 -0.011639853939414024,
 -0.007250551134347916,
 0.011669248342514038,
 0.015028521418571472,
 0.0037379509303718805,
 

In [35]:
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    limit=3
)

In [37]:
for i in results.points:
    print(i)

id=3 version=4 score=0.7600028 payload={'text': '\n        MongoDB is a document-oriented NoSQL database\n        that stores data using BSON documents.\n        ', 'title': 'MongoDB Basics', 'chapter': 3, 'course': 'Database Engineering'} vector=None shard_key=None order_value=None
id=5 version=6 score=0.563774 payload={'text': '\n        Retrieval augmented generation retrieves relevant\n        information before generating an LLM response.\n        ', 'title': 'RAG Fundamentals', 'chapter': 5, 'course': 'Generative AI'} vector=None shard_key=None order_value=None
id=2 version=5 score=0.53614074 payload={'text': '\n        Docker containers package applications together\n        with their dependencies.\n        ', 'title': 'Docker Fundamentals', 'chapter': 2, 'course': 'DevOps'} vector=None shard_key=None order_value=None
